# 歌词分词，词性标注

In [1]:
import json
import pandas as pd

# import jieba
# import jieba.posseg as pseg
import thulac
from collections import Counter
from openai import OpenAI

In [2]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [3]:
import sys
sys.path.append('..')

# 分词，词频与词性分析

In [4]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v'
}

In [5]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [15]:
thu = thulac.thulac(seg_only=False, filt=True) 

def process_lyrics_with_thulac(text, word_to_fix=None):
    if not text:
        return []
    
    # 2. 执行分词与词性标注
    # 返回格式为 [[word, pos], [word, pos], ...]
    words_with_pos = thu.cut(text)
    
    # 3. 过滤无意义字符与词性修正
    # thulac 的标点词性通常是 'w'
    filtered_data = []
    for word, pos in words_with_pos:
        word = word.strip()
        # 排除标点符号、空白字符
        if pos != 'w' and len(word) > 0:
            # 逻辑修正：word_to_fix 通常是修正词性
            if word_to_fix and word in word_to_fix:
                filtered_data.append((word, word_to_fix[word]))
            else:
                filtered_data.append((word, pos))
    
    # 4. 统计词频
    word_counts = Counter([item[0] for item in filtered_data])
    
    # 5. 汇总信息
    # 建立 word -> pos 映射
    word_pos_map = {word: pos for word, pos in filtered_data}
    
    sorted_results = []
    for word, count in word_counts.most_common():
        sorted_results.append({
            "word": word,
            "pos": word_pos_map[word],
            "freq": count
        })
    
    return sorted_results

Model loaded succeed


In [16]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'
    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    return df_word

In [17]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word['song_id'] = df_word['song_id'].astype(str)

    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# main

In [38]:
file_path_prefix = "data/jaychou/"
# file_path_prefix = "data/mayday/"
# file_path_prefix = "data/liuyuning/"

In [39]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,97773,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,000MkMni19ClKG,269,1059580800,晴天,8220,2003-07-31
1,102065756,004Z8Ihr0JIu5s,七里香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,七里香,003DFRzD192KKD,299,1091462400,七里香,20612,2004-08-03
2,449205,003aAYrm3GE0Ac,稻香,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,002Neh8l0uciQZ,223,1224000000,稻香,36062,2008-10-15
3,410316,002qU5aY3Qu24y,青花瓷,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,我很忙,002eFUFm2XYZ7z,239,1193932800,青花瓷,33021,2007-11-02
4,449198,003cI52o4daJJL,花海,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,魔杰座,002Neh8l0uciQZ,264,1224000000,花海,36062,2008-10-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,268352018,001glaI72k8BQX,Mojito,NaN,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,0009C3rp3Kfwg0,185,1591891200,Mojito,28791467,2022-07-14
158,213922043,0031TAKo0095np,不爱我就拉倒,NaN,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,001CnPE31iJ899,245,1526313600,不爱我就拉倒,28791467,2022-07-14
159,212877900,001J5QJL1pRQYB,等你下课 (with 杨瑞代),NaN,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,003bSL0v4bpKAx,270,1516204800,等你下课,28791467,2022-07-14
160,237773700,001qvvgF38HVc4,说好不哭 (with 五月天阿信),NaN,周杰伦,4558,0025NhlN2yWrP4,最伟大的作品,002gBTVk4JEE2T,222,1568646000,说好不哭,28791467,2022-07-14


In [40]:
# 五月天需要使用word_to_fix
if file_path_prefix == "data/mayday/":
    df_word = lyric_words_process(file_path_prefix, word_to_fix)
else:
    df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [41]:
df_word

,song_id,word,pos,freq
0,97773,好,a,10
1,97773,Si,v,7
2,97773,爱,v,6
3,97773,看,v,5
4,97773,见,v,5
...,...,...,...,...
13114,5105986,停止,v,1
13115,5105986,狼狈就,id,1
13116,5105986,让,v,1
13117,5105986,错,v,1


In [42]:
df_merged = words_data_merge(df_word, df_songs)
df_merged

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,97773,好,a,10,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,000MkMni19ClKG,269,1059580800,晴天,8220,2003-07-31
1,97773,Si,v,7,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,000MkMni19ClKG,269,1059580800,晴天,8220,2003-07-31
2,97773,爱,v,6,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,000MkMni19ClKG,269,1059580800,晴天,8220,2003-07-31
3,97773,看,v,5,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,000MkMni19ClKG,269,1059580800,晴天,8220,2003-07-31
4,97773,见,v,5,0039MnYb0qxYhV,晴天,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,周杰伦,4558,0025NhlN2yWrP4,叶惠美,000MkMni19ClKG,269,1059580800,晴天,8220,2003-07-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13114,5105986,停止,v,1,001xd0HI0X9GNq,一路向北,《头文字D》电影插曲,周杰伦,4558,0025NhlN2yWrP4,十一月的萧邦,002MAeob3zLXwZ,294,1119542400,一路向北,60671,2005-11-01
13115,5105986,狼狈就,id,1,001xd0HI0X9GNq,一路向北,《头文字D》电影插曲,周杰伦,4558,0025NhlN2yWrP4,十一月的萧邦,002MAeob3zLXwZ,294,1119542400,一路向北,60671,2005-11-01
13116,5105986,让,v,1,001xd0HI0X9GNq,一路向北,《头文字D》电影插曲,周杰伦,4558,0025NhlN2yWrP4,十一月的萧邦,002MAeob3zLXwZ,294,1119542400,一路向北,60671,2005-11-01
13117,5105986,错,v,1,001xd0HI0X9GNq,一路向北,《头文字D》电影插曲,周杰伦,4558,0025NhlN2yWrP4,十一月的萧邦,002MAeob3zLXwZ,294,1119542400,一路向北,60671,2005-11-01


In [43]:
df_merged.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

# 测试